# 💳 Loan Approval Prediction System
### Machine Learning Development Internship Task - Infobharat Interns (IBI)
**Author:** Nikhil Adari  
**Domain:** Machine Learning / Financial Underwriting  

---

## 📌 1. Project Overview & Objectives
The goal of this project is to build an end-to-end Machine Learning classification pipeline that accurately predicts whether a loan application should be **Approved (`1`)** or **Rejected (`0`)** based on demographic and financial metrics.

### Key Features:
- **`ApplicantID`**: Unique identification code
- **`Gender`**: Male / Female
- **`Age`**: Applicant age
- **`MonthlyIncome`**: Monthly income of the applicant ($)
- **`LoanAmount`**: Requested principal loan amount ($)
- **`CreditScore`**: FICO credit score (300 - 850)
- **`EmploymentStatus`**: Employed / Self-Employed / Unemployed
- **`ExistingLoans`**: Number of existing active loans
- **`LoanTerm`**: Loan duration in months
- **`PropertyArea`**: Urban / Semi-Urban / Rural
- **`LoanApproved`**: Target variable (0 = Rejected, 1 = Approved)


In [ ]:
# Imports & Configuration
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Plot settings
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

# Add src directory to path
sys.path.insert(0, os.path.abspath("../src"))
from preprocessing import LoanDataPreprocessor, clean_raw_data
from data_generator import generate_loan_data

---
## 📊 2. Data Loading & Inspection
We load the dataset containing 2,000 applicant records. Let's inspect its shape, summary statistics, and missing value distribution.


In [ ]:
data_path = os.path.join("..", "data", "loan_approval_dataset.csv")
if not os.path.exists(data_path):
    df_raw = generate_loan_data(n_samples=2000, random_state=42, inject_missing=True)
    os.makedirs(os.path.dirname(data_path), exist_ok=True)
    df_raw.to_csv(data_path, index=False)
else:
    df_raw = pd.read_csv(data_path)

print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
print("Missing Values per Column:")
print(df_raw.isnull().sum())

---
## 🔍 3. Exploratory Data Analysis (EDA) & Visualizations
We perform comprehensive visual exploration to extract insights into key drivers of loan approvals.


In [ ]:
# 1. Target Class Distribution (Approved vs Rejected)
plt.figure(figsize=(7, 5))
approval_counts = df_raw['LoanApproved'].value_counts()
labels = ['Approved (1)', 'Rejected (0)']
colors = ['#2ECC71', '#E74C3C']

plt.pie(approval_counts, labels=labels, colors=colors, autopct='%1.1f%%',
        startangle=140, explode=(0.05, 0), shadow=True, textprops={'fontsize': 12, 'fontweight': 'bold'})
plt.title("Loan Approval Overall Distribution", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# 2. Approvals by Employment Status
plt.figure(figsize=(8, 5))
emp_df = df_raw.dropna(subset=['EmploymentStatus', 'LoanApproved']).copy()
emp_df['ApprovalStatus'] = emp_df['LoanApproved'].map({1: 'Approved', 0: 'Rejected'})

sns.countplot(data=emp_df, x='EmploymentStatus', hue='ApprovalStatus', palette={'Approved': '#2ECC71', 'Rejected': '#E74C3C'}, edgecolor='black')
plt.title("Loan Approvals by Employment Status", fontsize=13, fontweight='bold')
plt.xlabel("Employment Status", fontweight='bold')
plt.ylabel("Applicant Count", fontweight='bold')
plt.show()

In [ ]:
# 3. Credit Score vs. Loan Approval
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_raw.dropna(subset=['CreditScore', 'LoanApproved']), x='LoanApproved', y='CreditScore', palette=['#E74C3C', '#2ECC71'])
plt.xticks([0, 1], ['Rejected (0)', 'Approved (1)'], fontweight='bold')
plt.title("Credit Score Distribution by Loan Approval Status", fontsize=13, fontweight='bold')
plt.xlabel("Decision", fontweight='bold')
plt.ylabel("Credit Score (FICO)", fontweight='bold')
plt.show()

In [ ]:
# 4. Correlation Heatmap
plt.figure(figsize=(9, 7))
numeric_cols = ["Age", "MonthlyIncome", "LoanAmount", "CreditScore", "ExistingLoans", "LoanTerm", "LoanApproved"]
corr = df_raw[numeric_cols].dropna().corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=13, fontweight='bold')
plt.show()

---
## ⚙️ 4. Data Preprocessing & Feature Engineering
We implement:
1. **Missing Value Imputation**: Median imputation for numerical features; Most frequent imputation for categorical features.
2. **Domain Feature Engineering**:
   - `EMIToIncomeRatio = (LoanAmount / LoanTerm) / MonthlyIncome`
   - `LoanToIncomeRatio = LoanAmount / MonthlyIncome`
3. **Categorical Encoding**: One-Hot Encoding for `Gender`, `EmploymentStatus`, `PropertyArea`.
4. **Feature Scaling**: `StandardScaler` fitted strictly on training data.


In [ ]:
df_cleaned = clean_raw_data(df_raw)
X = df_cleaned.drop(columns=['LoanApproved'])
y = df_cleaned['LoanApproved']

# 80/20 Stratified Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = LoanDataPreprocessor()
X_train = preprocessor.fit_transform(X_train_raw, y_train)
X_test = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names()

print(f"Transformed Features Count: {len(feature_names)}")
print("Engineered and Scaled Features:", feature_names)

---
## 🤖 5. Machine Learning Modeling & Comparison
We train and benchmark 3 classification models:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **Random Forest Classifier**


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_split=15, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_split=8, min_samples_leaf=4, random_state=42, n_jobs=1)
}

results = []
trained_models = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    cv_mean = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy").mean()
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc,
        "5-Fold CV Accuracy": cv_mean
    })

comparison_df = pd.DataFrame(results)
display(comparison_df.style.highlight_max(axis=0, color='#C6F6D5'))

---
## 🎯 6. Best Model Evaluation & Feature Importance
Let's analyze the performance of our winning model (**Random Forest Classifier**).


In [ ]:
best_model = trained_models["Random Forest"]
y_pred_best = best_model.predict(X_test)

print("Classification Report (Random Forest):\n")
print(classification_report(y_test, y_pred_best, target_names=["Rejected (0)", "Approved (1)"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Rejected', 'Predicted Approved'],
            yticklabels=['Actual Rejected', 'Actual Approved'])
plt.title("Random Forest Confusion Matrix", fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Feature Importance
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(10), x='Importance', y='Feature', palette='mako')
plt.title("Top 10 Feature Importances (Random Forest)", fontsize=13, fontweight='bold')
plt.xlabel("Importance Score", fontweight='bold')
plt.ylabel("Feature", fontweight='bold')
plt.show()

---
## 🧪 7. Real-Time Inference Demonstration
We test the model with sample applicant profiles to demonstrate real-time risk assessment.


In [ ]:
from predict import LoanPredictor

predictor = LoanPredictor(models_dir="../models")

applicant_sample = {
    "Gender": "Male",
    "Age": 36,
    "MonthlyIncome": 7500,
    "LoanAmount": 100000,
    "CreditScore": 740,
    "EmploymentStatus": "Employed",
    "ExistingLoans": 0,
    "LoanTerm": 180,
    "PropertyArea": "Semi-Urban"
}

decision = predictor.predict_single(applicant_sample)
print("=== APPLICATION ASSESSMENT ===")
print(f"Decision:             {decision['decision']}")
print(f"Approval Probability: {decision['approval_probability']:.2%}")
print(f"Risk Level:           {decision['risk_level']}")
print(f"Credit Tier:          {decision['credit_tier']}")
print(f"Estimated EMI:        ${decision['estimated_emi']:,.2f}")
print(f"EMI / Income Ratio:   {decision['emi_ratio_percent']:.1f}%")
print(f"Positive Drivers:     {decision['positives']}")
print(f"Risk Factors:         {decision['risks']}")

---
## 📋 8. Key Findings & Conclusion
1. **Primary Decision Factors**: `CreditScore`, `EMIToIncomeRatio` (debt burden), and `EmploymentStatus` are the strongest drivers of loan approval.
2. **Model Performance**: Random Forest achieved **~91.8% Accuracy** and **0.948 F1-Score**, showing strong generalization on test data.
3. **Production Readiness**: The model artifacts and preprocessing pipeline are modularized, serialized, and ready for deployment via CLI (`cli_app.py`) or Web UI (`app.py`).
